In [2]:
import importlib

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os

import model_functions
import utils

importlib.reload(model_functions)
importlib.reload(utils)

from model_functions import load_transition_matrix, gen_risk_matrix, gen_transition_probabilities, run_markov_cohort, \
    run_mc_sim, calculate_outcomes, plot_trace, load_costs, STAGE_ORDER, run_comparison

In [3]:
STAGE_ORDER = ['F0', 'F1', 'F2', 'F3', 'F4', 'HCC', 'DC', 'LT', 'post-LT', 'Death']
# TODO Add in 3_mo_post_LT, 6_mo_post_LT, 12_mo_post_LT??

MORTALITY_COLUMNS = ['Death']
# TODO split Death into liver-related, all-cause, cardiovascular?

DIR_HRP = "~/Documents/courses/HRP_392/"

In [16]:
def run_mc_sim_complex(base_transition_matrix, init_state, n_cycles, n_iter, starting_age=12, cycle_length=1., risk_matrix=None, begin_tx_at_cycle=0, tx_cycle_num=None, drug_stages=None):
    assert len(init_state) == len(STAGE_ORDER), "Initial state vector length must match number of states."
    assert len(init_state) == len(base_transition_matrix), "Transition matrix size must match number of states."

    if tx_cycle_num is None:
        tx_cycle_num = n_cycles

    markov_trace = np.zeros((n_iter, n_cycles + 1))
    on_drug = np.zeros((n_iter, n_cycles + 1))

    # keeping track of this assumes drug only has given efficacy over lifetime (two separate treatments of 36 months is similar to single tx of 72 months). Could change this to only track contiguous treatments.
    num_drug_cycles = np.zeros(n_iter)
    for i in range(n_iter):
        starting_state = np.random.choice(len(init_state), p=init_state)
        markov_trace[i, 0] = starting_state
        for t in range(n_cycles):
            age = starting_age + t * cycle_length

            local_risk_matrix = risk_matrix.copy()
            this_stage_idx = int(markov_trace[i, t])
            this_stage = STAGE_ORDER[this_stage_idx]
            # transform risk_matrix based on
            # drug stages
            local_on_drug = (((drug_stages is not None and this_stage in drug_stages) or
                              (drug_stages is None)) and
                             begin_tx_at_cycle <= t and
                             num_drug_cycles[i] < tx_cycle_num)
            if local_on_drug:
                on_drug[i, t] = 1
                num_drug_cycles[i] += 1
            else:
                local_risk_matrix = None

            t_matrix = gen_transition_probabilities(base_transition_matrix.copy(), np.floor(age),
                                                    os.path.join(DIR_HRP, "background_mortality.csv"),
                                                    local_risk_matrix)
            # print(t_matrix)
            markov_trace[i, t + 1] = np.random.choice(len(init_state), p=t_matrix.iloc[int(markov_trace[i, t]), :].values)

    trace_df = pd.DataFrame(markov_trace).T
    trace_df['age_float'] = np.linspace(starting_age, starting_age + n_cycles * cycle_length, n_cycles + 1)
    trace_df['age'] = np.floor(trace_df['age_float'])
    trace_df.set_index(['age_float', 'age'], inplace=True)

    on_drug_df = pd.DataFrame(on_drug).T
    on_drug_df['age_float'] = np.linspace(starting_age, starting_age + n_cycles * cycle_length, n_cycles + 1)
    on_drug_df['age'] = np.floor(on_drug_df['age_float'])
    on_drug_df.set_index(['age_float', 'age'], inplace=True)

    return trace_df.T, on_drug_df.T

In [17]:
cycle_len = 1/13
n_cycles = 63 * 13
cohort_size = 5
tm = load_transition_matrix('~/Documents/courses/HRP_392/final_baseline_transition_probs.csv', "_obs", cycle_length=cycle_len)
rm = gen_risk_matrix(regression_risk=1.56, progression_risk=1)

initial_prevalence = [.628, 0.309, 0.035, 0.016, 0.012, 0,0,0,0,0]

# treatment starting at 12, 72 weeks of treatment
mc_trace, drug_trace = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=18, drug_stages=['F2', 'F3'])

In [18]:
mc_trace

age_float,12.000000,12.076923,12.153846,12.230769,12.307692,12.384615,12.461538,12.538462,12.615385,12.692308,...,74.307692,74.384615,74.461538,74.538462,74.615385,74.692308,74.769231,74.846154,74.923077,75.000000
age,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,...,74.0,74.0,74.0,74.0,74.0,74.0,74.0,74.0,74.0,75.0
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0


In [19]:
drug_trace

age_float,12.000000,12.076923,12.153846,12.230769,12.307692,12.384615,12.461538,12.538462,12.615385,12.692308,...,74.307692,74.384615,74.461538,74.538462,74.615385,74.692308,74.769231,74.846154,74.923077,75.000000
age,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,...,74.0,74.0,74.0,74.0,74.0,74.0,74.0,74.0,74.0,75.0
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
drug_trace.sum(axis=1)

0     0.0
1    18.0
2    18.0
3     0.0
4     0.0
dtype: float64